<a href="https://colab.research.google.com/github/addisonk6/Python-Assignments/blob/main/Week15Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import gradio as gr
from datetime import datetime, timedelta
import json
import os

DATA_FILE = "sleep_data.json"

# Safe load
def load_data():
    if os.path.exists(DATA_FILE):
        try:
            with open(DATA_FILE, "r") as f:
                return json.load(f)
        except:
            return []
    return []

sleep_data = load_data()

def validate_time(t):
    try:
        datetime.strptime(t, "%H:%M")
        return True
    except:
        return False

def calculate_sleep(start, end):
    start_time = datetime.strptime(start, "%H:%M")
    end_time = datetime.strptime(end, "%H:%M")

    if end_time <= start_time:
        end_time += timedelta(days=1)

    duration = end_time - start_time
    return round(duration.total_seconds() / 3600, 2)

def add_sleep(sleep, wake):
    sleep = sleep.strip()
    wake = wake.strip()

    if not validate_time(sleep) or not validate_time(wake):
        return "❌ Invalid format. Use HH:MM (24-hour time)"

    hours = calculate_sleep(sleep, wake)

    entry = {
        "sleep": sleep,
        "wake": wake,
        "hours": hours
    }

    sleep_data.append(entry)

    try:
        with open(DATA_FILE, "w") as f:
            json.dump(sleep_data, f)
    except:
        return "❌ Error saving data"

    avg = round(sum(e["hours"] for e in sleep_data) / len(sleep_data), 2)

    output = "\n".join(
        [f"🛏 {e['sleep']} → {e['wake']} | {e['hours']} hrs" for e in sleep_data]
    )

    return f"{output}\n\n📊 Average Sleep: {avg} hrs"


# Show confirmation UI
def show_confirm():
    return gr.update(visible=True), gr.update(visible=True)


# Clear data with strict confirmation
def clear_data(confirm):
    global sleep_data

    if confirm != "YES":
        return (
            "⚠️ You must type YES exactly to confirm",
            gr.update(),
            gr.update()
        )

    sleep_data = []

    if os.path.exists(DATA_FILE):
        os.remove(DATA_FILE)

    return (
        "🗑 All data cleared",
        gr.update(visible=False, value=""),
        gr.update(visible=False)
    )


# UI
with gr.Blocks() as app:
    gr.Markdown("# 😴 Sleep Tracker")

    sleep_input = gr.Textbox(label="Sleep Time (HH:MM)")
    wake_input = gr.Textbox(label="Wake Time (HH:MM)")

    output = gr.Textbox(label="Results")

    with gr.Row():
        add_btn = gr.Button("Add Entry")
        clear_btn = gr.Button("Clear All Data")

    # Hidden confirmation UI
    confirm_input = gr.Textbox(
        label="Type YES to confirm clearing data",
        visible=False
    )

    confirm_btn = gr.Button("Confirm Clear", visible=False)

    # Button actions
    add_btn.click(add_sleep, inputs=[sleep_input, wake_input], outputs=output)

    clear_btn.click(
        show_confirm,
        outputs=[confirm_input, confirm_btn]
    )

    confirm_btn.click(
        clear_data,
        inputs=confirm_input,
        outputs=[output, confirm_input, confirm_btn]
    )

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://dbb78a84ce848c3d36.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
